# EduTune AI — Model Comparison

Compare persisted baseline and fine-tuned metrics when comparable results are available. Missing results are reported as unavailable rather than inferred.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

def load_jsonl(path):
    return pd.read_json(path, lines=True)

print("Project root:", PROJECT_ROOT)

E = PROJECT_ROOT / "data/evaluation"
def read_json(name):
    p = E / name
    if not p.exists():
        return None
    with open(p, encoding="utf-8") as f:
        return json.load(f)

baseline = read_json("baseline_evaluation_report.json")
comparison = read_json("model_comparison_report.json")
baseline_metrics = (baseline or {}).get("metrics", {})
print("Baseline metrics:", baseline_metrics)
print("Comparison artifact available:", comparison is not None)


## Persisted Comparison Artifact

In [ ]:
if comparison is not None:
    print(json.dumps(comparison, indent=2))
else:
    print("No persisted model comparison artifact is available.")


## Improvement Framework

In [ ]:
fine = {}
if isinstance(comparison, dict):
    fine = comparison.get("fine_tuned_metrics", comparison.get("finetuned_metrics", {})) or {}

rows = []
for metric in ["exact_match", "token_overlap"]:
    b, f = baseline_metrics.get(metric), fine.get(metric)
    rows.append({
        "metric": metric,
        "baseline": b,
        "fine_tuned": f,
        "absolute_improvement": None if b is None or f is None else f - b,
        "relative_improvement": None if b in (None, 0) or f is None else (f - b) / b
    })
display(pd.DataFrame(rows))


## Conclusion
Only comparable, persisted metrics are used for model comparison. No fine-tuned performance is fabricated when an artifact is missing.